In [6]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import repeat

from dropout import DropoutNd


def sample_gumbel_like(x: torch.Tensor) -> torch.Tensor:
    """Sample standard Gumbel noise with same shape/device/dtype as x."""
    # -log(-log(U))
    u = torch.rand_like(x).clamp_(1e-6, 1.0 - 1e-6)
    return -torch.log(-torch.log(u))


class TokenTopKRouter(nn.Module):
    """
    Token-level router.
    Input:  u of shape (B, H, L)
    Output: alpha/logits of shape (B, L, E)
    """

    def __init__(self, d_model: int, n_experts: int, top_k: int):
        super().__init__()
        self.n_experts = n_experts
        self.top_k = top_k
        self.proj = nn.Conv1d(d_model, n_experts, kernel_size=1)

    def forward(
        self,
        u: torch.Tensor,
        temperature: float = 1.0,
        straight_through: bool = True,
    ):
        """
        u: (B, H, L)
        returns:
          alpha:  (B, L, E) sparse convex weights
          logits: (B, L, E)
        """
        logits = self.proj(u).transpose(1, 2)  # (B, L, E)

        # Deterministic top-k at eval, noisy top-k during training
        if self.training:
            scores = (logits + sample_gumbel_like(logits)) / max(temperature, 1e-6)
        else:
            scores = logits

        soft = F.softmax(scores, dim=-1)  # dense convex weights

        k = min(self.top_k, self.n_experts)
        top_idx = scores.topk(k, dim=-1).indices                      # (B, L, k)
        mask = torch.zeros_like(scores).scatter_(-1, top_idx, 1.0)    # (B, L, E)

        # Sparse convex weights over selected top-k
        hard = soft * mask
        hard = hard / hard.sum(dim=-1, keepdim=True).clamp_min(1e-9)

        if straight_through:
            alpha = hard + soft - soft.detach()
        else:
            alpha = hard

        return alpha, logits


class TokenRoutedS4D(nn.Module):
    """
    Token-routed diagonal SSM.
    This is no longer FFT-kernel S4D because token-level routing makes the model time-varying.

    Main differences from minimal S4D:
      - explicit B is reintroduced
      - token-level router chooses a top-k convex combination of experts
      - computation is recurrent / scan-based
      - optional chunked processing carries state across chunks

    Interface:
      Input:
        transposed=True  -> u: (B, H, L)
        transposed=False -> u: (B, L, H)
      Output:
        y with same layout convention
    """

    def __init__(
        self,
        d_model: int,
        d_state: int = 64,
        n_experts: int = 4,
        top_k: int = 2,
        dropout: float = 0.0,
        transposed: bool = True,
        dt_min: float = 1e-3,
        dt_max: float = 1e-1,
        learnable_A_imag: bool = True,
    ):
        super().__init__()

        assert d_state % 2 == 0, "d_state must be even for complex conjugate parameterization."

        self.h = d_model
        self.n = d_state
        self.n2 = d_state // 2
        self.d_output = d_model
        self.transposed = transposed
        self.n_experts = n_experts
        self.top_k = top_k

        E, H, N2 = n_experts, d_model, self.n2

        # Token router
        self.router = TokenTopKRouter(H, E, top_k)

        # -------- Expert banks for SSM parameters --------
        # dt expert bank
        log_dt = torch.rand(E, H) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)
        self.log_dt = nn.Parameter(log_dt)  # (E, H)

        # A expert bank (diagonal complex)
        log_A_real = torch.log(0.5 * torch.ones(E, H, N2))
        self.log_A_real = nn.Parameter(log_A_real)  # (E, H, N2)

        A_imag = math.pi * repeat(torch.arange(N2, dtype=torch.float32), "n -> e h n", e=E, h=H)
        if learnable_A_imag:
            self.A_imag = nn.Parameter(A_imag)      # (E, H, N2)
        else:
            self.register_buffer("A_imag", A_imag)

        # Explicit B expert bank (complex)
        # Initialize close to "ones" because the original minimal S4D effectively folds input coupling into C.
        B = torch.ones(E, H, N2, dtype=torch.cfloat)
        self.B = nn.Parameter(torch.view_as_real(B))  # (E, H, N2, 2)

        # C expert bank (complex)
        C = torch.randn(E, H, N2, dtype=torch.cfloat)
        self.C = nn.Parameter(torch.view_as_real(C))  # (E, H, N2, 2)

        # D expert bank (real skip)
        self.D = nn.Parameter(torch.randn(E, H))      # (E, H)

        # -------- Pointwise output transform expert bank --------
        # Equivalent to Conv1d(H, 2H, kernel_size=1) + GLU
        self.out_W = nn.Parameter(torch.empty(E, 2 * H, H))  # (E, 2H, H)
        self.out_b = nn.Parameter(torch.empty(E, 2 * H))     # (E, 2H)

        # Nonlinearity / dropout
        self.activation = nn.GELU()
        self.dropout = DropoutNd(dropout) if dropout > 0.0 else nn.Identity()

        self.reset_parameters()

    def reset_parameters(self):
        E = self.n_experts
        for e in range(E):
            nn.init.kaiming_uniform_(self.out_W[e], a=math.sqrt(5))
            fan_in = self.out_W[e].size(-1)
            bound = 1.0 / math.sqrt(fan_in)
            nn.init.uniform_(self.out_b[e], -bound, bound)

    # ---------------- Mixing helpers ----------------

    @staticmethod
    def _mix_eh(alpha: torch.Tensor, bank: torch.Tensor) -> torch.Tensor:
        alpha = alpha.to(bank.dtype)
        return torch.einsum("bte,eh->bth", alpha, bank)

    @staticmethod
    def _mix_ehn(alpha: torch.Tensor, bank: torch.Tensor) -> torch.Tensor:
        alpha = alpha.to(bank.dtype)
        return torch.einsum("bte,ehn->bthn", alpha, bank)

    @staticmethod
    def _mix_eoh(alpha: torch.Tensor, bank: torch.Tensor) -> torch.Tensor:
        alpha = alpha.to(bank.dtype)
        return torch.einsum("bte,eoh->btoh", alpha, bank)

    @staticmethod
    def _mix_eo(alpha: torch.Tensor, bank: torch.Tensor) -> torch.Tensor:
        alpha = alpha.to(bank.dtype)
        return torch.einsum("bte,eo->bto", alpha, bank)

    @staticmethod
    def _mix_eo(alpha: torch.Tensor, bank: torch.Tensor) -> torch.Tensor:
        """
        alpha: (B, T, E)
        bank:  (E, O)
        out:   (B, T, O)
        """
        return torch.einsum("bte,eo->bto", alpha, bank)

    # ---------------- Materialization ----------------

    def _materialize_expert_banks(self):
        """
        Materialize expert banks in actual parameter space.
        returns:
          dt_bank: (E, H)          real
          A_bank:  (E, H, N2)      complex
          B_bank:  (E, H, N2)      complex
          C_bank:  (E, H, N2)      complex
          D_bank:  (E, H)          real
        """
        dt_bank = torch.exp(self.log_dt)                                # (E, H)
        A_bank = -torch.exp(self.log_A_real) + 1j * self.A_imag         # (E, H, N2)
        B_bank = torch.view_as_complex(self.B)                          # (E, H, N2)
        C_bank = torch.view_as_complex(self.C)                          # (E, H, N2)
        D_bank = self.D                                                 # (E, H)
        return dt_bank, A_bank, B_bank, C_bank, D_bank

    # ---------------- Core scan ----------------

    def _scan_chunk(
        self,
        u_chunk: torch.Tensor,
        alpha_chunk: torch.Tensor,
        state: torch.Tensor,
        dt_bank: torch.Tensor,
        A_bank: torch.Tensor,
        B_bank: torch.Tensor,
        C_bank: torch.Tensor,
        D_bank: torch.Tensor,
    ):
        """
        Process one chunk recurrently.

        u_chunk:     (B, H, T)
        alpha_chunk: (B, T, E)
        state:       (B, H, N2) complex

        returns:
          y_chunk: (B, H, T)
          state:   updated state, (B, H, N2)
        """
        Bsz, H, T = u_chunk.shape

        # Mix the banks for the whole chunk once; this is much faster than mixing per token.
        dt = self._mix_eh(alpha_chunk, dt_bank)     # (B, T, H)
        A = self._mix_ehn(alpha_chunk, A_bank)      # (B, T, H, N2)
        Bp = self._mix_ehn(alpha_chunk, B_bank)     # (B, T, H, N2)
        C = self._mix_ehn(alpha_chunk, C_bank)      # (B, T, H, N2)
        D = self._mix_eh(alpha_chunk, D_bank)       # (B, T, H)

        ys = []
        for t in range(T):
            u_t = u_chunk[:, :, t]      # (B, H)
            dt_t = dt[:, t, :]          # (B, H)
            A_t = A[:, t, :, :]         # (B, H, N2)
            B_t = Bp[:, t, :, :]        # (B, H, N2)
            C_t = C[:, t, :, :]         # (B, H, N2)
            D_t = D[:, t, :]            # (B, H)

            # Discretize current token's continuous-time diagonal SSM
            Abar_t = torch.exp(A_t * dt_t.unsqueeze(-1))  # (B, H, N2)

            # Numerically stable denominator for near-zero A
            A_denom = torch.where(
                A_t.abs() < 1e-6,
                A_t + (1e-6 + 0.0j),
                A_t,
            )
            Bbar_t = ((Abar_t - 1.0) / A_denom) * B_t     # (B, H, N2)

            # Recurrent state update
            state = Abar_t * state + Bbar_t * u_t.unsqueeze(-1)

            # Readout
            y_t = 2.0 * torch.einsum("bhn,bhn->bh", C_t, state).real + D_t * u_t
            ys.append(y_t)

        y_chunk = torch.stack(ys, dim=-1)  # (B, H, T)
        return y_chunk, state

    def _dynamic_output_projection(
        self,
        y_chunk: torch.Tensor,
        alpha_chunk: torch.Tensor,
    ):
        """
        Expert-mixed pointwise output projection + GLU.

        y_chunk:     (B, H, T)
        alpha_chunk: (B, T, E)

        returns:
          y_out: (B, H, T)
        """
        y_chunk = self.dropout(self.activation(y_chunk))    # (B, H, T)
        y_t = y_chunk.transpose(1, 2)                       # (B, T, H)

        W = self._mix_eoh(alpha_chunk, self.out_W)          # (B, T, 2H, H)
        b = self._mix_eo(alpha_chunk, self.out_b)           # (B, T, 2H)

        z = torch.einsum("btoh,bth->bto", W, y_t) + b       # (B, T, 2H)
        a, g = z.chunk(2, dim=-1)
        y_out = (a * torch.sigmoid(g)).transpose(1, 2)      # (B, H, T)
        return y_out

    # ---------------- Forward ----------------

    def forward(
        self,
        u: torch.Tensor,
        temperature: float = 1.0,
        chunk_size: int = None,
        straight_through: bool = True,
        return_router_aux: bool = False,
        **kwargs,
    ):
        """
        Args:
          u:
            transposed=True  -> (B, H, L)
            transposed=False -> (B, L, H)
          temperature:
            Gumbel router temperature during training
          chunk_size:
            None or <=0 means process full sequence at once.
            Positive int means recurrent chunking with state carry.
          straight_through:
            use ST estimator for top-k routing
          return_router_aux:
            return routing logits/weights

        Returns:
          y, aux_or_none
        """
        if not self.transposed:
            u = u.transpose(-1, -2)   # -> (B, H, L)

        Bsz, H, L = u.shape
        if chunk_size is None or chunk_size <= 0:
            chunk_size = L

        # Token-level routing over full sequence
        alpha, logits = self.router(
            u,
            temperature=temperature,
            straight_through=straight_through,
        )  # alpha/logits: (B, L, E)

        # Materialize expert banks once
        dt_bank, A_bank, B_bank, C_bank, D_bank = self._materialize_expert_banks()

        # Complex recurrent state
        state = torch.zeros(
            Bsz, H, self.n2,
            device=u.device,
            dtype=torch.cfloat,
        )

        outputs = []
        for start in range(0, L, chunk_size):
            end = min(start + chunk_size, L)

            u_chunk = u[:, :, start:end]          # (B, H, T)
            alpha_chunk = alpha[:, start:end, :]  # (B, T, E)

            y_chunk, state = self._scan_chunk(
                u_chunk=u_chunk,
                alpha_chunk=alpha_chunk,
                state=state,
                dt_bank=dt_bank,
                A_bank=A_bank,
                B_bank=B_bank,
                C_bank=C_bank,
                D_bank=D_bank,
            )

            y_chunk = self._dynamic_output_projection(y_chunk, alpha_chunk)
            outputs.append(y_chunk)

        y = torch.cat(outputs, dim=-1)  # (B, H, L)

        if not self.transposed:
            y = y.transpose(-1, -2)

        if return_router_aux:
            aux = {
                "routing_weights": alpha,   # (B, L, E)
                "routing_logits": logits,   # (B, L, E)
            }
            return y, aux

        return y, None


# ---------------- Example ----------------

if __name__ == "__main__":
    torch.manual_seed(0)

    model = TokenRoutedS4D(
        d_model=64,
        d_state=64,
        n_experts=8,
        top_k=2,
        dropout=0.1,
        transposed=True,
    )

    x = torch.randn(2, 64, 256)  # (B, H, L)

    # Training mode: noisy ST Gumbel top-k
    model.train()
    y, aux = model(
        x,
        temperature=1.0,
        chunk_size=64,
        straight_through=True,
        return_router_aux=True,
    )
    print("train:", y.shape, aux["routing_weights"].shape)

    # Eval mode: deterministic top-k
    model.eval()
    with torch.no_grad():
        y, aux = model(
            x,
            temperature=1.0,
            chunk_size=64,
            straight_through=True,
            return_router_aux=True,
        )
    print("eval :", y.shape, aux["routing_weights"].shape)

train: torch.Size([2, 64, 256]) torch.Size([2, 256, 8])
eval : torch.Size([2, 64, 256]) torch.Size([2, 256, 8])
